# CLaRa: Continuous Latent Reasoning — Full-Paper Reproduction (Kaggle T4/P100)

> **Paper:** *CLaRa: Bridging Retrieval and Generation with Continuous Latent Reasoning*  
> He et al., Apple / University of Edinburgh — February 2026  
> **Code:** https://github.com/apple/ml-clara

---

## Overview

CLaRa is an end-to-end Retrieval-Augmented Generation (RAG) framework that addresses two key limitations of classical RAG:

1. **Efficiency** — Documents are compressed into compact *memory tokens* (16×–256× compression) once and reused for both retrieval and generation, eliminating redundant text processing.
2. **Optimization** — Retrieval and generation are jointly trained via a differentiable top-k selection (Straight-Through estimator), allowing generator gradients to directly update the retriever.

### Two-Stage Training Pipeline

| Stage | Name | What trains | Loss | Purpose |
|-------|------|-------------|------|---------|
| **Stage I** | SCP — Salient Compressor Pretraining | `compressor` + `generator` LoRA | $\mathcal{L}_{CE} + \lambda \cdot \mathcal{L}_{MSE}$ | Learn to compress documents into memory tokens that preserve salient semantics |
| **Stage II** | E2E — End-to-End Joint Training | `query` + `generator` LoRA | $\mathcal{L}_{NTP}$ (next-token prediction) | Differentiable top-k retrieval + joint optimisation via ST estimator |

### This Notebook: Two Experimental Flows

```
Flow 1 ─── Train from Scratch (HotpotQA)
           Stage I  →  Stage II  →  Evaluate (HotpotQA)
           Purpose: Reproduce the full paper pipeline end-to-end.

Flow 2 ─── Transfer Learning from Apple Pretrained Weights
           Download E2E checkpoint → Convert → Fine-tune Stage II
           Evaluate on:
             (a) Apple pretrained model   (zero-shot, no fine-tuning)
             (b) Fine-tuned model         (TriviaQA + SQuAD separately)
           Purpose: Leverage Apple's SCP pretraining; demonstrate transfer.
```

### Paper Hyperparameters (Appendix B.4, Table 10)

| Hyperparameter | Paper Value | This Notebook |
|----------------|-------------|---------------|
| Base model | Mistral-7B-Instruct-v0.2 | Same |
| LoRA rank (r) | 16 | Same |
| LoRA alpha | 32 | Same |
| LoRA dropout | 0.1 | Same |
| Stage I LR | 2e-4 | Same |
| Stage II LR | 5e-6 | Same |
| Compression ratio | 16× (n_memory=16, doc_len=256) | Same |
| top-k documents | 5 | 2 (T4 VRAM limit) |
| Candidates | 20 | 8 (T4 VRAM limit) |
| Epochs | 1 | 1 |
| Warmup ratio | 0.03 | Same |
| ST temperature τ | — | 0.7 |

---

**Note on T4 adaptations:** The paper trains on 8×H100 GPUs. On a single T4 (16 GB), we reduce `num_candidates=8` and `top_k=2`. All other architectural choices are faithful to the paper.

---
##  Section 0 — Environment Setup

**Run once.** Clone the repo and install dependencies, then **restart the kernel** before running any further cells.

In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 0-A  │  Clone repository & install dependencies
# Run ONCE. After this cell completes → Session > Restart & Run All (skip cell 0-A).
# ═══════════════════════════════════════════════════════════════════════════════

import subprocess, sys

REPO_URL  = "https://github.com/Duy-Tuyen/introml-clara-implementation.git"
REPO_NAME = "introml-clara-implementation"
REPO_ROOT = f"/kaggle/working/{REPO_NAME}"

print("[1/3] Cloning repository...")
subprocess.run(["rm", "-rf", REPO_ROOT], check=True)
subprocess.run(["git", "clone", "-b", "feature/hieu2", REPO_URL, REPO_ROOT], check=True)

print("[2/3] Installing dependencies...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r",
     f"{REPO_ROOT}/requirements.txt", "-q"],
    check=True,
)

print("[3/3] Running setup script...")
subprocess.run([sys.executable, f"{REPO_ROOT}/setup_env.py"], check=True)

print("\n Setup complete. Please RESTART the kernel before continuing.")

[1/3] Cloning repository...


Cloning into '/kaggle/working/introml-clara-implementation'...


[2/3] Installing dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.2 MB/s eta 0:00:00
   ━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.3.1 which is incompatible.
torchvision 0.25.0+cu128 requires torch==2.10.0, but you have torch 2.3.1 which is incompatible.


[3/3] Running setup script...
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 27.5 MB/s eta 0:00:00

Patched bitsandbytes CUDA 12.8 → libbitsandbytes_cuda124_nocublaslt.so
Torch: 2.3.1+cu121 | CUDA: 12.1
GPU: Tesla T4
Môi trường Kaggle đã sẵn sàng. Hãy RESTART KERNEL rồi chạy tiếp.

 Setup complete. Please RESTART the kernel before continuing.


In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 0-B  │  Working directory & path configuration
# Run this cell FIRST after every kernel restart.
# ═══════════════════════════════════════════════════════════════════════════════

import os, sys

REPO_ROOT = "/kaggle/working/introml-clara-implementation"
assert os.path.isdir(REPO_ROOT), (
    f"Repository not found at {REPO_ROOT}. "
    "Please run Cell 0-A first, then restart the kernel."
)

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print(f"Working directory : {os.getcwd()}")
print(f"Python path entry : {sys.path[0]}")

Working directory : /kaggle/working/introml-clara-implementation
Python path entry : /kaggle/working/introml-clara-implementation


In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 0-C  │  GPU / VRAM diagnostics
# ═══════════════════════════════════════════════════════════════════════════════

import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Enable GPU: Settings > Accelerator > T4 or P100.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU   : {gpu_name}")
print(f"VRAM  : {vram_gb:.1f} GB")
print(f"CUDA  : {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")

if vram_gb < 14:
    print("\n Warning: Less than 14 GB VRAM. Consider reducing doc_max_length or batch_size.")
else:
    print("\n VRAM looks sufficient for T4-friendly config (num_candidates=8, top_k=2).")

GPU   : Tesla T4
VRAM  : 15.6 GB
CUDA  : 12.1
PyTorch: 2.3.1+cu121

 VRAM looks sufficient for T4-friendly config (num_candidates=8, top_k=2).


---
##  Section 1 — Smoke Test (64 samples, ~5 min)

Validates the full pipeline end-to-end before committing to a long training run.  
Runs Stage I → Stage II → Evaluate on HotpotQA with tiny data.  
**Skip this section if you're confident the environment is working.**

In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1  │  Smoke test — Stage I + II + Eval on 64/32 samples
# ═══════════════════════════════════════════════════════════════════════════════

import os, subprocess

SMOKE_CKPT = "/kaggle/working/clara-ckpts-smoke"

smoke_env = os.environ.copy()
smoke_env.update({
    "CLARA_DATASET"    : "hotpotqa",
    "CLARA_N_TRAIN"    : "64",
    "CLARA_N_VAL"      : "32",
    "CLARA_GRAD_ACC"   : "1",
    "CLARA_EVAL_BS"    : "2",
    "CLARA_OUTPUT_DIR" : SMOKE_CKPT,
})

print("═" * 60)
print("SMOKE TEST — Stage I (SCP)")
print("  Dataset       : hotpotqa")
print("  Train samples : 64  |  Val samples : 32")
print("  Loss          : L_CE + 0.1 * L_MSE")
print("═" * 60)
subprocess.run(["python", "-m", "scripts.train_stage1"],
               env=smoke_env, check=True)

print("\n" + "═" * 60)
print("SMOKE TEST — Stage II (End-to-End)")
print("  Differentiable retrieval with ST estimator")
print("═" * 60)
smoke_env["CLARA_STAGE1_DIR"] = f"{SMOKE_CKPT}/stage1_ep1"
subprocess.run(["python", "-m", "scripts.train_stage2"],
               env=smoke_env, check=True)

print("\n" + "═" * 60)
print("SMOKE TEST — Evaluation")
print("  Metrics: Exact Match (EM) & F1 (SQuAD-style normalisation)")
print("═" * 60)
smoke_env["CLARA_STAGE2_DIR"] = f"{SMOKE_CKPT}/stage2_ep1"
subprocess.run(["python", "-m", "scripts.evaluate"],
               env=smoke_env, check=True)

print("\n Smoke test passed — pipeline is working correctly.")

════════════════════════════════════════════════════════════
SMOKE TEST — Stage I (SCP)
  Dataset       : hotpotqa
  Train samples : 64  |  Val samples : 32
  Loss          : L_CE + 0.1 * L_MSE
════════════════════════════════════════════════════════════


Generating validation split: 100%|██████████| 7405/7405 [00:00<00:00, 41568.95 examples/s]


[hotpotqa|train] 64 samples  (eval_mode=oracle)
[hotpotqa|validation] 32 samples  (eval_mode=oracle)


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
2026-04-29 18:51:58.916513: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777488719.099522      97 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777488719.149245      97 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has al


Epoch 1/1  train=3.7148  val=2.9075  2.8/15.6GB
 Saved → /kaggle/working/clara-ckpts-smoke/stage1_ep1  (val=2.9075)


════════════════════════════════════════════════════════════
SMOKE TEST — Stage II (End-to-End)
  Differentiable retrieval with ST estimator
════════════════════════════════════════════════════════════


Loading checkpoint shards: 100%|██████████| 3/3 [00:37<00:00, 12.40s/it]
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/introml-clara-implementation/scripts/train_stage2.py", line 191, in <module>
    main()
  File "/kaggle/working/introml-clara-implementation/scripts/train_stage2.py", line 183, in main
    _load_stage1(model, cfg.stage1_ckpt_dir)
  File "/kaggle/working/introml-clara-implementation/scripts/train_stage2.py", line 44, in _load_stage1
    comp_weights = load_peft_weights(comp_dir)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py", line 415, in load_peft_weights
    has_remote_safetensors_file = file_exists(
                                  ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py", line 106, in _inner_fn
    validate_repo_id(a

CalledProcessError: Command '['python', '-m', 'scripts.train_stage2']' returned non-zero exit status 1.

---
## Flow 1 — Train from Scratch on HotpotQA

Full two-stage training pipeline on HotpotQA (multi-hop QA benchmark).

**Why HotpotQA instead of the paper's Qwen-32B synthetic data?**  
The paper synthesises SCP pretraining data using a locally-deployed Qwen-32B model on 2M Wikipedia documents — infeasible on a single T4. HotpotQA provides real QA supervision with multi-hop context, making it a practical substitute.

```
HotpotQA training set: 90,185 samples (Table 9 in paper)
HotpotQA eval set   :  7,384 samples
```

### Stage I — Salient Compressor Pretraining (SCP)

Trains `compressor` + `generator` LoRA adapters.  
Objective: $\mathcal{L}_{SCP} = \mathcal{L}_{CE} + \lambda \cdot \mathcal{L}_{MSE}$  
The MSE term aligns memory token representations with document hidden states, enforcing semantic fidelity.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2  │  Flow 1 — Stage I: Salient Compressor Pretraining (SCP)
#
# Paper §2.2: "Only the compressor LoRA θ_c is trained via cross-entropy loss.
# The generator is jointly trained to produce answers from memory tokens + query."
# Loss: L_CE + λ * L_MSE  (λ = 0.1, paper Appendix B.4)
# LR: 2e-4  |  Epochs: 1  |  Warmup: 3%
# ═══════════════════════════════════════════════════════════════════════════════

import os, subprocess

FLOW1_CKPT = "/kaggle/working/clara-ckpts-flow1"

flow1_stage1_env = os.environ.copy()
flow1_stage1_env.update({
    "CLARA_DATASET"    : "hotpotqa",
    "CLARA_N_TRAIN"    : "8000",      # Reduce to 4000 if OOM
    "CLARA_N_VAL"      : "500",
    "CLARA_OUTPUT_DIR" : FLOW1_CKPT,
})

print("╔" + "═" * 60 + "╗")
print("║  FLOW 1 — Stage I: Salient Compressor Pretraining (SCP)     ║")
print("╠" + "═" * 60 + "╣")
print("║  Dataset       : HotpotQA (distractor split)                 ║")
print("║  Train samples : 8,000  |  Val samples: 500                  ║")
print("║  Adapters      : compressor + generator (LoRA r=16)          ║")
print("║  Loss          : L_CE + 0.1 × L_MSE                         ║")
print("║  LR            : 2e-4 (cosine decay, 3% warmup)              ║")
print("║  Compression   : 16×  (doc_len=256 → 16 memory tokens)       ║")
print("╚" + "═" * 60 + "╝")

subprocess.run(
    ["python", "-m", "scripts.train_stage1"],
    env=flow1_stage1_env,
    check=True,
)

print(f"\n Stage I complete. Checkpoint saved to: {FLOW1_CKPT}/stage1_ep1")

### Stage II — End-to-End Joint Training

Initialises the `query` adapter from Stage I compressor weights, then jointly trains `query` + `generator` via differentiable top-k retrieval.

**Key mechanism (paper §3):** The Straight-Through (ST) estimator enables gradient flow from the generator loss back through the discrete top-k selection to the query encoder — the core innovation of CLaRa.

$Z = Z_{\text{hard}} + (Z_{\text{soft}} - \text{SG}(Z_{\text{soft}}))$

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3  │  Flow 1 — Stage II: End-to-End Joint Training
#
# Paper §3: Joint training of query reasoner θ_qr and generator θ_g
# via NTP loss with differentiable top-k selection (ST estimator).
# Query adapter initialised from compressor weights (paper §3, init strategy).
# LR: 5e-6  |  Epochs: 1  |  top_k=2 (paper uses 5, reduced for T4)
# ═══════════════════════════════════════════════════════════════════════════════

import os, subprocess

FLOW1_CKPT      = "/kaggle/working/clara-ckpts-flow1"
FLOW1_STAGE1_DIR = f"{FLOW1_CKPT}/stage1_ep1"

assert os.path.isdir(FLOW1_STAGE1_DIR), (
    f"Stage I checkpoint not found at {FLOW1_STAGE1_DIR}. "
    "Please run Cell 2 (Stage I training) first."
)

flow1_stage2_env = os.environ.copy()
flow1_stage2_env.update({
    "CLARA_DATASET"    : "hotpotqa",
    "CLARA_N_TRAIN"    : "8000",
    "CLARA_N_VAL"      : "500",
    "CLARA_STAGE1_DIR" : FLOW1_STAGE1_DIR,
    "CLARA_OUTPUT_DIR" : FLOW1_CKPT,
})

print("╔" + "═" * 60 + "╗")
print("║  FLOW 1 — Stage II: End-to-End Joint Training                ║")
print("╠" + "═" * 60 + "╣")
print("║  Dataset       : HotpotQA (distractor, multi-hop)            ║")
print("║  Train samples : 8,000  |  Val samples: 500                  ║")
print("║  Adapters      : query + generator (LoRA r=16)               ║")
print("║  Retrieval     : differentiable top-k (ST estimator, τ=0.7)  ║")
print("║  Candidates    : 8  |  top_k selected: 2                     ║")
print("║  Loss          : NTP (next-token prediction only)            ║")
print("║  LR            : 5e-6 (cosine decay, 3% warmup)              ║")
print("║  Init          : query ← compressor weights (Stage I)        ║")
print("╚" + "═" * 60 + "╝")

subprocess.run(
    ["python", "-m", "scripts.train_stage2"],
    env=flow1_stage2_env,
    check=True,
)

print(f"\n Stage II complete. Checkpoint saved to: {FLOW1_CKPT}/stage2_ep1")

### Evaluation — Flow 1 Model (HotpotQA)

Evaluates the from-scratch model using SQuAD-style EM and F1.  
Paper reference (Oracle setting, CLaRa-Mistral-7B 16×): **EM ≈ 56.76%, F1 ≈ 69.57%** on HotpotQA (Table 2, instruction-tuned init).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4  │  Flow 1 — Evaluate model trained from scratch on HotpotQA
#
# Metrics (paper B.3):
#   EM  — Exact Match: % predictions exactly matching gold answer
#         (after SQuAD normalisation: lowercase, strip punct/articles)
#   F1  — Token-level F1: harmonic mean of precision and recall
# ═══════════════════════════════════════════════════════════════════════════════

import os, subprocess

FLOW1_CKPT = "/kaggle/working/clara-ckpts-flow1"
FLOW1_STAGE2_DIR = f"{FLOW1_CKPT}/stage2_ep1"

assert os.path.isdir(FLOW1_STAGE2_DIR), (
    f"Stage II checkpoint not found at {FLOW1_STAGE2_DIR}. "
    "Please run Cell 3 (Stage II training) first."
)

flow1_eval_env = os.environ.copy()
flow1_eval_env.update({
    "CLARA_DATASET"       : "hotpotqa",
    "CLARA_EVAL_MODE"     : "oracle",    # Oracle = gold doc in candidate pool
    "CLARA_EVAL_BS"       : "4",
    "CLARA_N_VAL"         : "500",
    "CLARA_STAGE2_DIR"    : FLOW1_STAGE2_DIR,
    "CLARA_MODEL_VERSION" : "Flow1_ScratchTrained_HotpotQA",
})

print("╔" + "═" * 60 + "╗")
print("║  FLOW 1 — Evaluation: From-Scratch Model on HotpotQA         ║")
print("╠" + "═" * 60 + "╣")
print("║  Model   : Trained from scratch (Flow 1)                     ║")
print("║  Dataset : HotpotQA  |  Eval mode: oracle                    ║")
print("║  Metrics : Exact Match (EM) + F1 (SQuAD normalisation)       ║")
print("║  Paper ref (oracle, 16×): EM~56.8%  F1~69.6%  [Table 2]     ║")
print("╚" + "═" * 60 + "╝")

subprocess.run(
    ["python", "-m", "scripts.evaluate"],
    env=flow1_eval_env,
    check=True,
)

print("\n Results appended to: results/eval_scores.csv")

---
##  Flow 2 — Transfer Learning from Apple Pretrained Weights

This flow leverages Apple's publicly released E2E checkpoint (`apple/CLaRa-7B-E2E` on Hugging Face), which was trained by Apple using the full SCP pipeline on 2M Wikipedia documents with Qwen-32B synthesis.

**Two distinct model comparisons:**

| Model | Description | Training |
|-------|-------------|----------|
| **Model A** (Pretrained, zero-shot) | Apple E2E checkpoint, no fine-tuning | Apple's SCP + E2E on full Wikipedia |
| **Model B** (Fine-tuned) | Apple weights → fine-tuned Stage II on TriviaQA + SQuAD | Flow 2 fine-tuning |

**Step 1** — Download Apple E2E checkpoint  
**Step 2** — Convert to this repo's adapter format (query/generator split)  
**Step 3a** — Evaluate Apple pretrained model directly (zero-shot)  
**Step 3b** — Fine-tune Stage II on TriviaQA and SQuAD separately  
**Step 4** — Evaluate fine-tuned models

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5  │  Flow 2 — Step 1: Download Apple pretrained E2E checkpoint
#
# Repo: apple/CLaRa-7B-E2E on Hugging Face
# Contains: adapters.pth or lora/ directory + optional decoder_first_last_layers.pth
# ═══════════════════════════════════════════════════════════════════════════════

import subprocess, os

PRETRAINED_RAW = "/kaggle/working/pretrained-apple-e2e"

print("╔" + "═" * 60 + "╗")
print("║  FLOW 2 — Step 1: Download Apple CLaRa-7B-E2E Checkpoint     ║")
print("╠" + "═" * 60 + "╣")
print("║  Source: apple/CLaRa-7B-E2E (Hugging Face)                   ║")
print(f"║  Target: {PRETRAINED_RAW[:48]}  ║")
print("╚" + "═" * 60 + "╝")

subprocess.run([
    "python", "-m", "scripts.download_pretrained",
    "--repo", "apple/CLaRa-7B-E2E",
    "--out",  PRETRAINED_RAW,
    "--revision", "compression-16",
], check=True)

print(f"\n Download complete: {PRETRAINED_RAW}")
print("Files downloaded:")
for f in os.listdir(PRETRAINED_RAW):
    print(f"  • {f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 6  │  Flow 2 — Step 2: Convert Apple E2E checkpoint to adapter format
#
# Apple's checkpoint stores weights as a single adapters.pth or PEFT lora/.
# Our repo expects separate query/ and generator/ adapter directories.
# convert_apple_e2e.py: loads Apple weights → splits into query + generator
#   adapters + saves mem_token_embed as clara_stage2_extra.pth
# ═══════════════════════════════════════════════════════════════════════════════

import subprocess, os

PRETRAINED_RAW       = "/kaggle/working/pretrained-apple-e2e"
PRETRAINED_CONVERTED = "/kaggle/working/pretrained-apple-e2e-converted"

assert os.path.isdir(PRETRAINED_RAW), (
    f"Pretrained checkpoint not found at {PRETRAINED_RAW}. Run Cell 5 first."
)

print("╔" + "═" * 60 + "╗")
print("║  FLOW 2 — Step 2: Convert to Stage II Adapter Format         ║")
print("╠" + "═" * 60 + "╣")
print("║  Input  (Apple format) : adapters.pth or lora/ directory     ║")
print("║  Output (this repo)    : adapters/query/ + adapters/gen/     ║")
print("║                          clara_stage2_extra.pth              ║")
print("╚" + "═" * 60 + "╝")

subprocess.run([
    "python", "-m", "scripts.convert_apple_e2e",
    "--input",  PRETRAINED_RAW,
    "--output", PRETRAINED_CONVERTED,
], check=True)

print(f"\n Conversion complete: {PRETRAINED_CONVERTED}")
print("Output structure:")
for root, dirs, files in os.walk(PRETRAINED_CONVERTED):
    level = root.replace(PRETRAINED_CONVERTED, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  • {f}")

### Model A — Evaluate Apple Pretrained Checkpoint (Zero-Shot)

Evaluates the Apple pretrained weights **directly**, without any fine-tuning on TriviaQA or SQuAD.  
This represents the upper bound of Apple's SCP pretraining quality.  

Paper reference (Table 2, Normal, CLaRa-Mistral-7B 16×):  
- TriviaQA is not a paper benchmark; NQ is comparable — **EM ≈ 41.02%, F1 ≈ 50.89%**

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 7  │  Flow 2 — Step 3a: Evaluate Apple pretrained model (zero-shot)
#            Datasets: TriviaQA + SQuAD  (no fine-tuning)
# ═══════════════════════════════════════════════════════════════════════════════

import os, subprocess

PRETRAINED_CONVERTED = "/kaggle/working/pretrained-apple-e2e-converted"

assert os.path.isdir(PRETRAINED_CONVERTED), (
    f"Converted checkpoint not found at {PRETRAINED_CONVERTED}. Run Cell 6 first."
)

for dataset in ["triviaqa", "squad"]:
    print("\n" + "╔" + "═" * 60 + "╗")
    print(f"║  MODEL A (Pretrained) — Zero-Shot Eval: {dataset.upper():<21}║")
    print("╠" + "═" * 60 + "╣")
    print("║  Checkpoint : Apple CLaRa-7B-E2E (no fine-tuning)           ║")
    print(f"║  Dataset    : {dataset:<47}║")
    print("║  Eval mode  : oracle  |  Metrics: EM + F1                   ║")
    print("╚" + "═" * 60 + "╝")

    eval_env = os.environ.copy()
    eval_env.update({
        "CLARA_DATASET"       : dataset,
        "CLARA_EVAL_MODE"     : "oracle",
        "CLARA_EVAL_BS"       : "4",
        "CLARA_N_VAL"         : "500",
        "CLARA_STAGE2_DIR"    : PRETRAINED_CONVERTED,
        "CLARA_MODEL_VERSION" : f"ModelA_ApplePretrained_{dataset}",
    })

    subprocess.run(
        ["python", "-m", "scripts.evaluate"],
        env=eval_env,
        check=True,
    )

print("\ Apple pretrained evaluation complete (both datasets).")
print(" Results appended to: results/eval_scores.csv")

### Model B — Fine-Tune Stage II (Apple Init → TriviaQA & SQuAD)

Starting from the converted Apple checkpoint, fine-tunes Stage II (`query` + `generator`) on each dataset separately.  
This is the transfer learning experiment: Apple's strong SCP representations + domain adaptation.

Note: `CLARA_STAGE1_DIR` is still required by `train_stage2.py` (for structural init), but `CLARA_STAGE2_INIT` overrides with Apple weights.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 8  │  Flow 2 — Step 3b: Fine-tune Stage II (Apple init → TriviaQA & SQuAD)
#
# For each dataset:
#   1. Load Apple E2E converted checkpoint as init
#   2. Fine-tune query + generator LoRA adapters (LR=5e-6, 1 epoch)
#   3. Save best checkpoint by validation loss
#
# This is the "Instruction-tuned-initialized" setting from Table 2.
# ═══════════════════════════════════════════════════════════════════════════════

import os, subprocess

PRETRAINED_CONVERTED = "/kaggle/working/pretrained-apple-e2e-converted"

# Flow 1 Stage I checkpoint — required by train_stage2.py for architecture setup
# If Flow 1 was not run, set this to any existing Stage I checkpoint or create
# a minimal one with the smoke test (SMOKE_CKPT/stage1_ep1)
STAGE1_FOR_INIT = (
    "/kaggle/working/clara-ckpts-flow1/stage1_ep1"
    if os.path.isdir("/kaggle/working/clara-ckpts-flow1/stage1_ep1")
    else "/kaggle/working/clara-ckpts-smoke/stage1_ep1"
)

assert os.path.isdir(PRETRAINED_CONVERTED), (
    f"Converted checkpoint not found at {PRETRAINED_CONVERTED}. Run Cell 6 first."
)
assert os.path.isdir(STAGE1_FOR_INIT), (
    f"No Stage I checkpoint found. Run Flow 1 (Cell 2) or Smoke Test (Cell 1) first."
)

FT_CKPTS = {}  # dataset → checkpoint path

for dataset in ["triviaqa", "squad"]:
    out_dir = f"/kaggle/working/clara-ckpts-ft-{dataset}"
    FT_CKPTS[dataset] = f"{out_dir}/stage2_ep1"

    print("\n" + "╔" + "═" * 60 + "╗")
    print(f"║  FLOW 2 — Fine-Tune Stage II: {dataset.upper():<30}║")
    print("╠" + "═" * 60 + "╣")
    print("║  Init       : Apple CLaRa-7B-E2E converted weights         ║")
    print("║  Adapters   : query + generator (LoRA r=16)                ║")
    print(f"║  Dataset    : {dataset:<46}║")
    print("║  Train size : 8,000  |  Val size: 500                      ║")
    print("║  LR         : 5e-6 (cosine decay)  |  Epochs: 1            ║")
    print(f"║  Output     : {out_dir[:46]}║")
    print("╚" + "═" * 60 + "╝")

    ft_env = os.environ.copy()
    ft_env.update({
        "CLARA_DATASET"    : dataset,
        "CLARA_N_TRAIN"    : "8000",
        "CLARA_N_VAL"      : "500",
        "CLARA_STAGE1_DIR" : STAGE1_FOR_INIT,
        "CLARA_STAGE2_INIT": PRETRAINED_CONVERTED,  # Apple init overrides
        "CLARA_OUTPUT_DIR" : out_dir,
    })

    subprocess.run(
        ["python", "-m", "scripts.train_stage2"],
        env=ft_env,
        check=True,
    )

    print(f"\n Fine-tuning on {dataset} complete → {FT_CKPTS[dataset]}")

print("\n All fine-tuning runs complete.")

### Model B — Evaluate Fine-Tuned Models (TriviaQA & SQuAD)

Evaluates both fine-tuned models. Results are saved to `results/eval_scores.csv` alongside Model A (pretrained).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 9  │  Flow 2 — Step 4: Evaluate fine-tuned models
#            Datasets: TriviaQA + SQuAD (Model B — fine-tuned)
# ═══════════════════════════════════════════════════════════════════════════════

import os, subprocess

for dataset in ["triviaqa", "squad"]:
    ft_ckpt = f"/kaggle/working/clara-ckpts-ft-{dataset}/stage2_ep1"

    assert os.path.isdir(ft_ckpt), (
        f"Fine-tuned checkpoint not found at {ft_ckpt}. Run Cell 8 first."
    )

    print("\n" + "╔" + "═" * 60 + "╗")
    print(f"║  MODEL B (Fine-tuned) — Eval: {dataset.upper():<30}║")
    print("╠" + "═" * 60 + "╣")
    print("║  Checkpoint : Apple init → fine-tuned on dataset            ║")
    print(f"║  Dataset    : {dataset:<46}║")
    print("║  Eval mode  : oracle  |  Metrics: EM + F1                   ║")
    print("╚" + "═" * 60 + "╝")

    eval_env = os.environ.copy()
    eval_env.update({
        "CLARA_DATASET"       : dataset,
        "CLARA_EVAL_MODE"     : "oracle",
        "CLARA_EVAL_BS"       : "4",
        "CLARA_N_VAL"         : "500",
        "CLARA_STAGE2_DIR"    : ft_ckpt,
        "CLARA_MODEL_VERSION" : f"ModelB_FineTuned_{dataset}",
    })

    subprocess.run(
        ["python", "-m", "scripts.evaluate"],
        env=eval_env,
        check=True,
    )

print("\n✅ Fine-tuned model evaluation complete (both datasets).")
print("📄 All results saved to: results/eval_scores.csv")

---
##  Section 4 — Results Summary

Aggregates all evaluation results from `results/eval_scores.csv` and displays a formatted comparison table.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 10  │  Results summary — all evaluation runs
# ═══════════════════════════════════════════════════════════════════════════════

import os
import pandas as pd

CSV_PATH = "results/eval_scores.csv"

if not os.path.exists(CSV_PATH):
    print(f"No results found at {CSV_PATH}. Run evaluation cells first.")
else:
    df = pd.read_csv(CSV_PATH)

    print("═" * 80)
    print("  CLaRa EXPERIMENT RESULTS SUMMARY")
    print("═" * 80)
    print()

    # Format display
    display_df = df[[
        'Model_Version', 'Dataset', 'Eval_Mode',
        'Exact_Match(%)', 'F1_Score(%)', 'Timestamp'
    ]].copy()

    # Sort for readability
    display_df = display_df.sort_values(['Dataset', 'Model_Version'])

    pd.set_option('display.max_colwidth', 45)
    pd.set_option('display.width', 120)
    print(display_df.to_string(index=False))

    print()
    print("─" * 80)
    print("PAPER REFERENCE (Table 2 — Oracle, CLaRa-Mistral-7B 16×):")
    print("  NQ        : EM=63.29%  F1=71.54%")
    print("  HotpotQA  : EM=57.54%  F1=71.17%")
    print("  (Instruction-tuned init, Normal setting)")
    print("─" * 80)
    print()
    print("NOTE: Our results are expected to be lower due to:")
    print("  • Single T4 GPU (paper: 8×H100)")
    print("  • Reduced num_candidates=8 vs 20 in paper")
    print("  • top_k=2 vs 5 in paper")
    print("  • HotpotQA substitute for Qwen-32B synthetic SCP data")
    print("  • 8,000 training samples vs full dataset in paper")

---
##  Section 5 — Qualitative Inference Examples

Runs interactive inference to inspect the model's predictions qualitatively.  
Change `TEST_CASES` or `CKPT_TO_USE` to test different models.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 11  │  Qualitative inference — inspect model predictions
# ═══════════════════════════════════════════════════════════════════════════════

import os, sys, torch
from peft import load_peft_weights_local, set_peft_model_state_dict
from configs.config import CLaRaConfig
from models.clara_model import build_clara_model

# ── Configure which checkpoint to use ────────────────────────────────────────
# Options:
#   Flow 1 model     : "/kaggle/working/clara-ckpts-flow1/stage2_ep1"
#   Apple pretrained : "/kaggle/working/pretrained-apple-e2e-converted"
#   Fine-tuned TriviaQA: "/kaggle/working/clara-ckpts-ft-triviaqa/stage2_ep1"
#   Fine-tuned SQuAD   : "/kaggle/working/clara-ckpts-ft-squad/stage2_ep1"

CKPT_TO_USE = "/kaggle/working/clara-ckpts-ft-triviaqa/stage2_ep1"

TEST_CASES = [
    {
        'doc': 'The Battle of Hastings was fought on 14 October 1066 between '
               'the Norman-French army of William, the Duke of Normandy, and '
               'an English army under the Anglo-Saxon King Harold Godwinson.',
        'q'  : 'When was the Battle of Hastings fought?',
        'expected': '14 October 1066',
    },
    {
        'doc': 'Weldenia is a monotypic genus of flowering plants in the family '
               'Commelinaceae, native to Mexico and Guatemala.',
        'q'  : 'Which genus grows originally in Mexico and Guatemala, '
               'Phylica or Weldenia?',
        'expected': 'Weldenia',
    },
    {
        'doc': 'Albert Einstein was born on 14 March 1879 in Ulm, in the '
               'Kingdom of Württemberg in the German Empire. He developed the '
               'theory of relativity.',
        'q'  : 'Where was Albert Einstein born?',
        'expected': 'Ulm',
    },
]

# ── Load model ────────────────────────────────────────────────────────────────
print(f"Loading checkpoint: {CKPT_TO_USE}")
assert os.path.isdir(CKPT_TO_USE), f"Checkpoint not found: {CKPT_TO_USE}"

cfg = CLaRaConfig()
model, tokenizer = build_clara_model(cfg)

query_dir = os.path.join(CKPT_TO_USE, 'adapters', 'query')
gen_dir   = os.path.join(CKPT_TO_USE, 'adapters', 'generator')
extra_pth = os.path.join(CKPT_TO_USE, 'clara_stage2_extra.pth')

if os.path.isdir(query_dir):
    set_peft_model_state_dict(model.backbone,
                              load_peft_weights_local(query_dir), adapter_name='query')
if os.path.isdir(gen_dir):
    set_peft_model_state_dict(model.backbone,
                              load_peft_weights_local(gen_dir), adapter_name='generator')
if os.path.exists(extra_pth):
    model.mem_token_embed.data = torch.load(extra_pth, map_location='cuda')['mem_token_embed']

print("✓ Checkpoint loaded.\n")

# ── Run inference ─────────────────────────────────────────────────────────────
model.eval()
print("═" * 60)
print("INFERENCE EXAMPLES")
print("═" * 60)

with torch.no_grad():
    for i, t in enumerate(TEST_CASES, 1):
        doc_enc = tokenizer(
            [t['doc']], max_length=cfg.doc_max_length,
            padding='max_length', truncation=True, return_tensors='pt')
        q_enc = tokenizer(
            f"[INST] {t['q']} [/INST]", max_length=cfg.max_qa_len,
            padding='max_length', truncation=True, return_tensors='pt')

        doc_ids  = doc_enc['input_ids'].unsqueeze(0).cuda()
        doc_mask = doc_enc['attention_mask'].unsqueeze(0).cuda()
        cand_mask = torch.ones(1, 1, dtype=torch.long, device='cuda')
        q_ids    = q_enc['input_ids'].cuda()
        q_mask   = q_enc['attention_mask'].cuda()

        answer = model.generate_answer_e2e(
            doc_ids, doc_mask, cand_mask, q_ids, q_mask,
            max_new_tokens=32
        )[0]

        print(f"\n[{i}] Question : {t['q']}")
        print(f"    Expected : {t['expected']}")
        print(f"    Model    : {answer}")
        match = t['expected'].lower() in answer.lower()
        print(f"    Match    : {'✓ YES' if match else '✗ NO'}")

print("\n" + "═" * 60)

---
##  Experimental Notes & Reproducibility

### T4 GPU Adaptations vs. Paper

| Setting | Paper | This notebook | Reason |
|---------|-------|---------------|--------|
| `num_candidates` | 20 | 8 | T4 VRAM |
| `top_k` | 5 | 2 | T4 VRAM |
| SCP pretraining data | 2M Wiki docs via Qwen-32B | HotpotQA (8k samples) | No Qwen-32B access |
| Training GPUs | 8×H100 | 1×T4 | Kaggle free tier |
| Training samples | Full dataset | 8,000 | Runtime constraint |

### Flow 1 vs Flow 2 — Key Differences

| | Flow 1 (Scratch) | Flow 2 (Transfer) |
|-|------------------|-------------------|
| Stage I compressor | Trained on HotpotQA | Apple's (2M docs, Qwen-32B) |
| Stage II init | Flow 1 Stage I | Apple E2E converted |
| Fine-tune datasets | HotpotQA | TriviaQA, SQuAD |
| Expected quality | Lower (weak SCP) | Higher (strong SCP) |

### Expected Results (Oracle setting, 16× compression)

From paper Table 2 (instruction-tuned init, closest to Flow 2):

| Dataset | EM | F1 |
|---------|----|----|  
| NQ | 63.29% | 71.54% |
| HotpotQA | 57.54% | 71.17% |

Our results will be lower due to the T4 adaptations above. This is expected and documented.

### Citation

```bibtex
@article{he2026clara,
  title   = {CLaRa: Bridging Retrieval and Generation with Continuous Latent Reasoning},
  author  = {He, Jie and Bai, Richard He and Williamson, Sinead and Pan, Jeff Z. 
             and Jaitly, Navdeep and Zhang, Yizhe},
  journal = {arXiv preprint arXiv:2511.18659},
  year    = {2026}
}
```